In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [0]:
%sql

WITH source_dedup AS (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY AmazonOrderId, file_name
             ORDER BY LastUpdateDate DESC
           ) AS rn
    FROM ${catalog}.${schema}.amazon_orders_bronze
  )
  WHERE rn = 1
)

MERGE INTO ${catalog}.${schema}.amazon_orders_silver AS target
USING source_dedup AS source
ON target.AmazonOrderId = source.AmazonOrderId AND target.file_name = source.file_name

WHEN MATCHED THEN
  UPDATE SET
    target.PurchaseDate = to_timestamp(source.PurchaseDate),
    target.LastUpdateDate = to_timestamp(source.LastUpdateDate),
    target.OrderStatus = source.OrderStatus,
    target.FulfillmentChannel = source.FulfillmentChannel,
    target.SalesChannel = source.SalesChannel,
    target.ShipServiceLevel = source.ShipServiceLevel,
    target.NumberOfItemsShipped = source.NumberOfItemsShipped,
    target.NumberOfItemsUnshipped = source.NumberOfItemsUnshipped,
    target.PaymentMethod = source.PaymentMethod,
    target.IsReplacementOrder = source.IsReplacementOrder,
    target.MarketplaceId = source.MarketplaceId,
    target.ShipmentServiceLevelCategory = source.ShipmentServiceLevelCategory,
    target.OrderType = source.OrderType,
    target.EarliestShipDate = to_timestamp(source.EarliestShipDate),
    target.LatestShipDate = to_timestamp(source.LatestShipDate),
    target.EarliestDeliveryDate = to_timestamp(source.EarliestDeliveryDate),
    target.LatestDeliveryDate = to_timestamp(source.LatestDeliveryDate),
    target.IsBusinessOrder = source.IsBusinessOrder,
    target.IsPrime = source.IsPrime,
    target.IsGlobalExpressEnabled = source.IsGlobalExpressEnabled,
    target.IsPremiumOrder = source.IsPremiumOrder,
    target.IsSoldByAB = source.IsSoldByAB,
    target.IsIBA = source.IsIBA,
    target.IsISPU = source.IsISPU,
    target.IsAccessPointOrder = source.IsAccessPointOrder,
    target.EasyShipShipmentStatus = source.EasyShipShipmentStatus,
    target.ElectronicInvoiceStatus = source.ElectronicInvoiceStatus,
    target.file_name = source.file_name,
    target.OrderTotal_CurrencyCode = source.OrderTotal_CurrencyCode,
    target.OrderTotal_Amount = source.OrderTotal_Amount,
    target.DefaultShipFromLocationAddress_Name = source.DefaultShipFromLocationAddress_Name,
    target.DefaultShipFromLocationAddress_AddressLine1 = source.DefaultShipFromLocationAddress_AddressLine1,
    target.DefaultShipFromLocationAddress_City = source.DefaultShipFromLocationAddress_City,
    target.DefaultShipFromLocationAddress_StateOrRegion = source.DefaultShipFromLocationAddress_StateOrRegion,
    target.DefaultShipFromLocationAddress_PostalCode = source.DefaultShipFromLocationAddress_PostalCode,
    target.DefaultShipFromLocationAddress_CountryCode = source.DefaultShipFromLocationAddress_CountryCode,
    target.DefaultShipFromLocationAddress_Phone = source.DefaultShipFromLocationAddress_Phone,
    target.DefaultShipFromLocationAddress_AddressType = source.DefaultShipFromLocationAddress_AddressType,
    target.FulfillmentIntruction_FulfillmentSupplySourceId = source.FulfillmentIntruction_FulfillmentSupplySourceId,
    target.AutomatedShippingSettingsStatus = source.AutomatedShippingSettingsStatus,
    target.PaymentMethodDetail = source.PaymentMethodDetail,
    target.Md5_Hash = md5(concat_ws('', source.AmazonOrderId,
      to_timestamp(source.PurchaseDate),
      to_timestamp(source.LastUpdateDate),
      source.OrderStatus,
      source.FulfillmentChannel,
      source.SalesChannel,
      source.ShipServiceLevel,
      source.NumberOfItemsShipped,
      source.NumberOfItemsUnshipped,
      source.PaymentMethod,
      source.IsReplacementOrder,
      source.MarketplaceId,
      source.ShipmentServiceLevelCategory,
      source.OrderType,
      to_timestamp(source.EarliestShipDate),
      to_timestamp(source.LatestShipDate),
      to_timestamp(source.EarliestDeliveryDate),
      to_timestamp(source.LatestDeliveryDate),
      source.IsBusinessOrder,
      source.IsPrime,
      source.IsGlobalExpressEnabled,
      source.IsPremiumOrder,
      source.IsSoldByAB,
      source.IsIBA,
      source.IsISPU,
      source.IsAccessPointOrder,
      source.EasyShipShipmentStatus,
      source.ElectronicInvoiceStatus,
      source.file_name,
      source.OrderTotal_CurrencyCode,
      source.OrderTotal_Amount,
      source.DefaultShipFromLocationAddress_Name,
      source.DefaultShipFromLocationAddress_AddressLine1,
      source.DefaultShipFromLocationAddress_City,
      source.DefaultShipFromLocationAddress_StateOrRegion,
      source.DefaultShipFromLocationAddress_PostalCode,
      source.DefaultShipFromLocationAddress_CountryCode,
      source.DefaultShipFromLocationAddress_Phone,
      source.DefaultShipFromLocationAddress_AddressType,
      source.FulfillmentIntruction_FulfillmentSupplySourceId,
      source.AutomatedShippingSettingsStatus,
      source.PaymentMethodDetail))

WHEN NOT MATCHED THEN
  INSERT (
    AmazonOrderId,
    PurchaseDate,
    LastUpdateDate,
    OrderStatus,
    FulfillmentChannel,
    SalesChannel,
    ShipServiceLevel,
    NumberOfItemsShipped,
    NumberOfItemsUnshipped,
    PaymentMethod,
    IsReplacementOrder,
    MarketplaceId,
    ShipmentServiceLevelCategory,
    OrderType,
    EarliestShipDate,
    LatestShipDate,
    EarliestDeliveryDate,
    LatestDeliveryDate,
    IsBusinessOrder,
    IsPrime,
    IsGlobalExpressEnabled,
    IsPremiumOrder,
    IsSoldByAB,
    IsIBA,
    IsISPU,
    IsAccessPointOrder,
    EasyShipShipmentStatus,
    ElectronicInvoiceStatus,
    file_name,
    OrderTotal_CurrencyCode,
    OrderTotal_Amount,
    DefaultShipFromLocationAddress_Name,
    DefaultShipFromLocationAddress_AddressLine1,
    DefaultShipFromLocationAddress_City,
    DefaultShipFromLocationAddress_StateOrRegion,
    DefaultShipFromLocationAddress_PostalCode,
    DefaultShipFromLocationAddress_CountryCode,
    DefaultShipFromLocationAddress_Phone,
    DefaultShipFromLocationAddress_AddressType,
    FulfillmentIntruction_FulfillmentSupplySourceId,
    AutomatedShippingSettingsStatus,
    PaymentMethodDetail,
    Md5_Hash
  )
  VALUES (
    source.AmazonOrderId,
    to_timestamp(source.PurchaseDate),
    to_timestamp(source.LastUpdateDate),
    source.OrderStatus,
    source.FulfillmentChannel,
    source.SalesChannel,
    source.ShipServiceLevel,
    source.NumberOfItemsShipped,
    source.NumberOfItemsUnshipped,
    source.PaymentMethod,
    source.IsReplacementOrder,
    source.MarketplaceId,
    source.ShipmentServiceLevelCategory,
    source.OrderType,
    to_timestamp(source.EarliestShipDate),
    to_timestamp(source.LatestShipDate),
    to_timestamp(source.EarliestDeliveryDate),
    to_timestamp(source.LatestDeliveryDate),
    source.IsBusinessOrder,
    source.IsPrime,
    source.IsGlobalExpressEnabled,
    source.IsPremiumOrder,
    source.IsSoldByAB,
    source.IsIBA,
    source.IsISPU,
    source.IsAccessPointOrder,
    source.EasyShipShipmentStatus,
    source.ElectronicInvoiceStatus,
    source.file_name,
    source.OrderTotal_CurrencyCode,
    source.OrderTotal_Amount,
    source.DefaultShipFromLocationAddress_Name,
    source.DefaultShipFromLocationAddress_AddressLine1,
    source.DefaultShipFromLocationAddress_City,
    source.DefaultShipFromLocationAddress_StateOrRegion,
    source.DefaultShipFromLocationAddress_PostalCode,
    source.DefaultShipFromLocationAddress_CountryCode,
    source.DefaultShipFromLocationAddress_Phone,
    source.DefaultShipFromLocationAddress_AddressType,
    source.FulfillmentIntruction_FulfillmentSupplySourceId,
    source.AutomatedShippingSettingsStatus,
    source.PaymentMethodDetail,
    md5(concat_ws('', source.AmazonOrderId,
      to_timestamp(source.PurchaseDate),
      to_timestamp(source.LastUpdateDate),
      source.OrderStatus,
      source.FulfillmentChannel,
      source.SalesChannel,
      source.ShipServiceLevel,
      source.NumberOfItemsShipped,
      source.NumberOfItemsUnshipped,
      source.PaymentMethod,
      source.IsReplacementOrder,
      source.MarketplaceId,
      source.ShipmentServiceLevelCategory,
      source.OrderType,
      to_timestamp(source.EarliestShipDate),
      to_timestamp(source.LatestShipDate),
      to_timestamp(source.EarliestDeliveryDate),
      to_timestamp(source.LatestDeliveryDate),
      source.IsBusinessOrder,
      source.IsPrime,
      source.IsGlobalExpressEnabled,
      source.IsPremiumOrder,
      source.IsSoldByAB,
      source.IsIBA,
      source.IsISPU,
      source.IsAccessPointOrder,
      source.EasyShipShipmentStatus,
      source.ElectronicInvoiceStatus,
      source.file_name,
      source.OrderTotal_CurrencyCode,
      source.OrderTotal_Amount,
      source.DefaultShipFromLocationAddress_Name,
      source.DefaultShipFromLocationAddress_AddressLine1,
      source.DefaultShipFromLocationAddress_City,
      source.DefaultShipFromLocationAddress_StateOrRegion,
      source.DefaultShipFromLocationAddress_PostalCode,
      source.DefaultShipFromLocationAddress_CountryCode,
      source.DefaultShipFromLocationAddress_Phone,
      source.DefaultShipFromLocationAddress_AddressType,
      source.FulfillmentIntruction_FulfillmentSupplySourceId,
      source.AutomatedShippingSettingsStatus,
      source.PaymentMethodDetail))
  )


In [0]:
%sql
SELECT * FROM ${catalog}.${schema}.amazon_orders_silver
order by amazonorderid 